# Get Your Static IP & Verify It

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivikasavnish/algo-trading-notebooks/blob/main/notebooks/00_get_static_ip_and_verify.ipynb)

Claim a dedicated static IPv6 and confirm your egress IP actually changes.

Part 00 of 35 in the [ServLoci algo/options trading notebook series](https://comm.servloci.in/docs) — full index in `notebooks/README.md`.

## Setup

In [ ]:
# Get your dedicated static IPv6 + SOCKS5 credentials free:
#   https://comm.servloci.in/register        (or /auth/google?free=1 for an instant trial)
# Your api_key / api_secret pair shows up in the portal after signup:
#   https://comm.servloci.in/user
!pip install -q "requests[socks]"
!curl -sL https://comm.servloci.in/sdk/servloci.py -o servloci.py

import os
from servloci import ServLoci

SERVLOCI_API_KEY = os.environ.get("SERVLOCI_API_KEY", "dhan:1000000001")   # broker:client_id
SERVLOCI_API_SECRET = os.environ.get("SERVLOCI_API_SECRET", "")            # from the portal — leave blank to run this notebook in demo mode

sl = None
if SERVLOCI_API_SECRET:
    sl = ServLoci(api_key=SERVLOCI_API_KEY, api_secret=SERVLOCI_API_SECRET)
    print("ServLoci configured:", sl.host, sl.port)
else:
    print("SERVLOCI_API_SECRET not set — running in demo mode (no live proxy calls).")

## Why algo trading needs a static IP at all

Every Indian broker that exposes an order-placement API — Zerodha's Kite Connect,
DhanHQ, Groww's Trade API, FYERS — ties API keys to a small allowlist of source
IPs. This is not a broker being difficult: SEBI's algo-trading framework and each
exchange's colocation/API rules push brokers to bind trading credentials to a
known, auditable network origin, the same way a bank binds a corporate API key
to a fixed egress IP. An API key plus a secret is enough to authenticate; the IP
allowlist is the second control that limits *where* that authentication is
allowed to come from, so a leaked key alone can't be used to place orders from
an arbitrary machine.

That's exactly what makes a laptop, a home router, or a notebook running on
Google Colab unsuitable for this without help. Home and mobile ISPs mostly hand
out **dynamic** IPs that rotate on reconnect — the address your broker
allowlisted yesterday may not be the one you have today. Colab's outbound IP is
worse: it's drawn from a shared Google Cloud pool, reused across unrelated
users and sessions, and frequently already present on abuse blocklists, so even
if you could allowlist it, the next runtime restart would hand you a different
one anyway.

A **static IP** fixes the address side of the problem: it's an IP that's yours,
doesn't change between sessions, and can be allowlisted once. The cell below
proves the concept end to end — it prints Colab's own (unusable) address, then
the address your broker will actually see once traffic is routed through a
static IPv6 assigned to your account. That second address is your **egress
IP**: the IP the *destination server* (the broker's API) observes as the
origin of the request, after any proxying in front of it. Whitelisting is
always done against the egress IP, never against the IP your notebook happens
to be running on.

In [ ]:
import requests

direct_ip = requests.get("https://api.ipify.org?format=json", timeout=8).json()["ip"]
print("Direct egress IP (Colab's own — NOT yours):", direct_ip)

if sl:
    proxied_ip = sl.session().get("https://api.ipify.org?format=json", timeout=8).json()["ip"]
    print("Proxied egress IP (whitelist THIS with your broker):", proxied_ip)
    assert proxied_ip != direct_ip, "expected the proxy to change the egress IP"
else:
    print("Set SERVLOCI_API_SECRET above (get it free at https://comm.servloci.in/register), then re-run this cell.")

IPv6 rather than IPv4 matters here mainly for supply: IPv4 addresses are
scarce and expensive to dedicate one-per-user, while IPv6 has enough address
space to hand every account a permanent, non-shared address. Most Indian
broker API consoles accept IPv6 allowlist entries directly; a handful still
ask for a fallback IPv4, which is why some brokers (see notebooks 02-05)
support both.

Next: whitelist the printed IPv6 in your broker's API console (see notebooks
02-05 for the per-broker steps), then move on to the SDK quickstart to see how
that same address gets applied automatically to every broker call you make
from Python.

---

Next: [ServLoci SDK Quickstart](01_servloci_sdk_quickstart.ipynb) »

Try the concepts above interactively: [Options Strategy Builder](https://comm.servloci.in/tools/strategy-builder) · [Docs](https://comm.servloci.in/docs) · [Get your static IP](https://comm.servloci.in/register)